# HuggingFace basics

HuggingFace gives three levels of abstraction over the same models. `pipeline()` runs a whole task in one line. `load_dataset` loads standardised data. `AutoTokenizer` and `AutoModel` hand you the raw tokens and hidden states. You move down the ladder as you need more control.

This notebook is network heavy. Every download is wrapped so that it prints a skip note and continues when the hub is unavailable, which keeps the notebook reproducible offline.

In [ ]:
from common.dl import get_device, set_seed

set_seed(42)
device = get_device()
print(f"device: {device}")

## 1. pipeline: a task in one line

`pipeline("sentiment-analysis")` downloads a fine-tuned model, tokenises the input, runs the forward pass and decodes the label. We pin a small model (`distilbert-base-uncased-finetuned-sst-2-english`) so the download is light.

In [ ]:
from transformers import pipeline

sentences = [
    "The replacement airpods were refurbished, not genuine.",
    "The Goodguys price matched and the service was great.",
    "The volume in the left earbud was considerably lower.",
]

try:
    classifier = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
    )
    for sentence, out in zip(sentences, classifier(sentences)):
        print(f"{out['label']:<8} {out['score']:.3f}  {sentence}")
except (OSError, ImportError) as err:
    print(f"transformers hub unavailable offline, skipping pipeline demo ({type(err).__name__}).")

## 2. datasets: standardised data loading

`load_dataset` returns a splittable, typed table. We take 1% of the `rotten_tomatoes` train split, then look at the columns, the length and one row. Every dataset on the hub exposes the same interface, so the loading code does not change between tasks.

In [ ]:
from datasets import load_dataset

try:
    reviews = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train[:1%]")
    print(f"columns: {reviews.column_names}")
    print(f"rows: {len(reviews)}")
    example = reviews[0]
    print(f"label: {example['label']}")
    print(f"text: {example['text'][:90]}...")
except (OSError, ImportError, ValueError) as err:
    print(f"datasets hub unavailable offline, skipping dataset demo ({type(err).__name__}).")

## 3. tokenizer and model: raw tokens and hidden states

Below the pipeline sit two objects. The tokenizer maps text to `input_ids`, the integer indices the model reads. The model maps those ids to a hidden state, one vector per token. We use `distilbert-base-uncased` for both and keep everything in float32 on the detected device.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_id = "distilbert-base-uncased"
sentence = "HuggingFace makes transformers approachable."

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    encoded = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    print(f"input_ids: {encoded['input_ids'][0].tolist()}")
    print(f"tokens:    {tokens}")

    model = AutoModel.from_pretrained(model_id).to(device)
    inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model(**inputs)
    hidden = output.last_hidden_state
    print(f"hidden state shape: {tuple(hidden.shape)}")
    print(f"dtype: {hidden.dtype}")
except (OSError, ImportError) as err:
    print(f"transformers hub unavailable offline, skipping token and model demo ({type(err).__name__}).")

The hidden state shape is `(batch, seq, hidden)`. The batch is 1 because we passed one sentence. The seq length matches the token count, which includes the `[CLS]` and `[SEP]` markers DistilBERT adds. The hidden dimension is 768, the size of each token's contextual vector. These vectors are the input a classification head would learn from.

## 4. What breaks

The tokenizer and model must share the same id. The tokenizer's vocabulary and special tokens are tied to how that model was trained, so pairing `bert-base-uncased` ids with a `distilbert` model produces wrong, silent results.

`pipeline()` hides batching and truncation. For long or many inputs you often need to set `batch_size`, `truncation=True` and `max_length` yourself, which means dropping down to the tokenizer and model shown above.